Геометрия реальных изображений (≤ 55 минут)

**Цель:** за один запуск получить проверяемые числа и графики о том, помогают ли *геометрические признаки реальных изображений* отличать AI-generated изображения. Это **не воспроизведение REM** и не обучение LoRA. Используется **замороженный DINOv2-S** и четыре статистических метода:

1. **Mahalanobis** — гауссовское описание real-признаков.
2. **kNN** — расстояние до пяти ближайших настоящих изображений.
3. **Global PCA** — расстояние до одного линейного подпространства real-признаков.
4. **Local PCA** — расстояние до приближённого *локального касательного пространства* ближайшей real-окрестности.

Два независимых этапа:

- **A. Синтетическая окружность в R³²:** известное многообразие и контролируемые отклонения; результат ~секунды. Это **демонстрация геометрической идеи**, не детекция AI.
- **B. Tiny GenImage:** обучение статистики исключительно на real-изображениях. Отдельно тестируем неизвестные генераторы на clean и JPEG60. Получаем ROC-AUC, TPR при real-only калибровке 5% FPR, 95%-е bootstrap CI для clean.

**Настройки:** Kaggle → **Accelerator: T4 (или T4×2)**, **Internet: On**, добавьте Tiny GenImage через Add Input. Используем **только GPU0**: для маленькой модели это надёжнее, чем `DataParallel`, который ломал VAE в предыдущем ноутбуке. Никаких VAE, LoRA, многопроцессных DataLoader, `QueueFeederThread`.

**Бюджет:** soft-stop через 55 минут с проверкой между группами; каждый завершённый блок сразу записывает результаты в `/kaggle/working/geo_fast_55min/results/`. Если внешняя загрузка весов зависнет, ограничение не может её принудительно остановить. Запускать через **Run All** или **Save & Run All**; главное — **начинать с нуля**.

## 0. Установка и настройки

In [43]:
%pip -q install "transformers==4.48.3" "huggingface-hub>=0.28,<1" "scikit-learn>=1.4,<2" tqdm matplotlib pandas

Note: you may need to restart the kernel to use updated packages.


In [44]:
from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageOps, ImageFilter
from torch.utils.data import Dataset, DataLoader
from IPython.display import display, FileLink
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, roc_curve
import io, os, time, json, hashlib, random, zipfile, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# БЫСТРЫЙ ПИЛОТ. В этом режиме всё, кроме модели DINOv2, — мгновенные/короткие расчёты.
SEED = 42
TIME_LIMIT_MIN = 55
TRAIN_N = 200
CAL_N = 80
TEST_REAL_N = 80
FAKE_PER_GEN = 80
BATCH = 32
PCA_RANK = 32
LOCAL_K = 32             # ближайшие real для оценки локальной геометрии
LOCAL_RANK = 8           # число локальных касательных направлений
KNN_K = 5
GENERATORS_PRIORITY = ('BigGAN', 'SD1.5', 'Midjourney', 'VQDM', 'Wukong', 'ADM', 'GLIDE')
CONDITIONS = ('clean', 'jpeg60')
BOOTSTRAP = 200
MODEL_ID = 'facebook/dinov2-small'

ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/mnt/data')
OUT = WORK / 'geo_fast_55min' / 'results'
CACHE = WORK / 'geo_fast_55min' / 'cache'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
RUN_START = time.monotonic()

def elapsed_min():
    return (time.monotonic() - RUN_START) / 60

def budget_left():
    return TIME_LIMIT_MIN - elapsed_min()

def write_json(name, obj):
    (OUT/name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

def export_zip():
    path = WORK / 'geo_fast_55min_results.zip'
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in OUT.rglob('*'):
            if f.is_file(): z.write(f, f.relative_to(OUT))
    print('Результаты:', path)
    return path

CONFIG = {k:globals()[k] for k in ('SEED','TIME_LIMIT_MIN','TRAIN_N','CAL_N','TEST_REAL_N',
    'FAKE_PER_GEN','BATCH','PCA_RANK','LOCAL_K','LOCAL_RANK','KNN_K','GENERATORS_PRIORITY',
    'CONDITIONS','BOOTSTRAP','MODEL_ID')}
write_json('config.json',CONFIG)
print('PyTorch:',torch.__version__, 'CUDA:',torch.cuda.is_available(),
      'GPU count:',torch.cuda.device_count())
print('Ожидаемая верхняя граница рабочего бюджета:',TIME_LIMIT_MIN,'минут (soft-stop)')

PyTorch: 2.11.0+cu130 CUDA: True GPU count: 2
Ожидаемая верхняя граница рабочего бюджета: 55 минут (soft-stop)


## 1. Реализация четырёх методов — одинаковый код для синтетики и DINOv2

In [45]:
class GeometryScorers:
    def __init__(self, bank, p_global=32, local_k=32, p_local=8, knn_k=5):
        bank = np.asarray(bank,dtype=np.float32)
        self.bank = bank
        n,d = bank.shape
        assert n > max(p_global, local_k) and local_k > p_local >= 1
        self.global_pca = PCA(n_components=min(p_global,n-1,d), svd_solver='randomized',random_state=SEED)
        self.global_pca.fit(bank)
        self.maha = LedoitWolf().fit(bank.astype(np.float64))
        self.nn5 = NearestNeighbors(n_neighbors=knn_k,metric='euclidean').fit(bank)
        self.nn1 = NearestNeighbors(n_neighbors=1,metric='euclidean').fit(bank)
        self.local_nn = NearestNeighbors(n_neighbors=local_k,metric='euclidean').fit(bank)
        self.centers = np.empty((n,d),dtype=np.float32)
        self.bases = np.empty((n,d,p_local),dtype=np.float32)
        ids = self.local_nn.kneighbors(bank,return_distance=False)
        for i, row in enumerate(tqdm(ids,desc='Построение local PCA',leave=False)):
            X = bank[row].astype(np.float64)
            mu = X.mean(axis=0)
            Xc = X-mu
            # Спектр маленькой матрицы Gram K×K вместо SVD большого D×D.
            gram = Xc @ Xc.T
            vals, vecs = np.linalg.eigh(gram)
            vals, vecs = vals[::-1],vecs[:,::-1]
            p = p_local
            V = Xc.T @ (vecs[:,:p] / np.sqrt(np.maximum(vals[:p],1.e-12)))
            # При полном ненулевом ранге столбцы V ортонормированы.
            Q,_ = np.linalg.qr(V,mode='reduced')
            self.centers[i] = mu.astype(np.float32)
            self.bases[i] = Q.astype(np.float32)
        self.ratio = float(np.sum(self.global_pca.explained_variance_ratio_))

    def score(self, data):
        X = np.asarray(data,dtype=np.float32)
        if X.ndim==1: X=X[None,:]
        scores={}
        scores['Mahalanobis'] = self.maha.mahalanobis(X.astype(np.float64))
        scores['kNN'] = self.nn5.kneighbors(X,return_distance=True)[0].mean(axis=1)
        # Orthogonal distance to global affine PCA space (must center!).
        scores['Global_PCA'] = np.linalg.norm(X-self.global_pca.inverse_transform(self.global_pca.transform(X)),axis=1)
        closest = self.nn1.kneighbors(X,return_distance=False).ravel()
        delta = X-self.centers[closest]
        U = self.bases[closest]
        proj = np.einsum('bd,bdp->bp',delta,U,optimize=True)
        normal = delta-np.einsum('bp,bdp->bd',proj,U,optimize=True)
        scores['Local_PCA'] = np.linalg.norm(normal,axis=1)
        return scores

METHODS = ('Mahalanobis','kNN','Global_PCA','Local_PCA')

def auc_ci(real,fake,seed=SEED,B=BOOTSTRAP):
    a,b=np.asarray(real),np.asarray(fake)
    rng = np.random.default_rng(seed)
    aucs=np.empty(B)
    y=np.r_[np.zeros(len(a)),np.ones(len(b))]
    for t in range(B):
        rr=a[rng.integers(0,len(a),len(a))]
        ff=b[rng.integers(0,len(b),len(b))]
        aucs[t]=roc_auc_score(y,np.r_[rr,ff])
    return tuple(np.quantile(aucs,[.025,.975]))

def save_metric_plots(metrics):
    clean = metrics[metrics.condition=='clean']
    if clean.empty: return
    order=[g for g in GENERATORS_PRIORITY if g in set(clean.generator)]
    pivot=clean.pivot(index='generator',columns='method',values='roc_auc').reindex(order)
    pivot=pivot.reindex(columns=list(METHODS))
    fig,ax=plt.subplots(figsize=(10,4.7))
    mat=pivot.to_numpy()
    im=ax.imshow(mat,cmap='viridis',vmin=0,vmax=1,aspect='auto')
    ax.set_xticks(range(len(pivot.columns)),pivot.columns,rotation=20,ha='right')
    ax.set_yticks(range(len(pivot.index)),pivot.index)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j,i,f'{mat[i,j]:.3f}',ha='center',va='center',
                    color='white' if mat[i,j]<0.45 or mat[i,j]>0.75 else 'black',fontsize=10)
    ax.set_title('Tiny GenImage: ROC-AUC / clean (higher = better)')
    fig.colorbar(im,ax=ax,label='ROC-AUC')
    fig.tight_layout()
    fig.savefig(OUT/'auc_clean_heatmap.png',dpi=180)
    plt.close(fig)
    return pivot

Окружность с известной геометрией

In [46]:
# Никаких скачиваний, нейросетей и GPU. Это проверяет именно геометрическую мотивацию.
rng=np.random.default_rng(SEED)
D=32
A,_=np.linalg.qr(rng.normal(size=(D,2)))
base=rng.normal(size=(D,))*0.1

def ring(n,radius=1.0,noise=0.02):
    ang=rng.uniform(0,2*np.pi,n)
    xy=radius*np.stack([np.cos(ang),np.sin(ang)],axis=1)
    X=base+xy@A.T+rng.normal(0,noise,(n,D))
    return X.astype(np.float32),ang

X_train,ang_train=ring(400)
X_real,ang_real=ring(160)
X_anom,ang_anom=ring(160,radius=1.4)
toy=GeometryScorers(X_train,p_global=2,local_k=32,p_local=1,knn_k=5)
s_real=toy.score(X_real)
s_fake=toy.score(X_anom)
toy_rows=[dict(method=m,roc_auc=roc_auc_score(np.r_[np.zeros(160),np.ones(160)],
                                            np.r_[s_real[m],s_fake[m]])) for m in METHODS]
toy_metrics=pd.DataFrame(toy_rows)
toy_metrics.to_csv(OUT/'toy_geometry_metrics.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].scatter((X_train-base)@A[:,0],(X_train-base)@A[:,1],s=7,alpha=.4,label='train real')
axs[0].scatter((X_anom-base)@A[:,0],(X_anom-base)@A[:,1],s=7,alpha=.5,label='off-manifold')
axs[0].set_aspect('equal');axs[0].set_title('Toy: true 1-D circle in 32-D');axs[0].legend(fontsize=8)
axs[1].bar(toy_metrics.method,toy_metrics.roc_auc)
axs[1].set_ylim(0,1);axs[1].set_ylabel('ROC-AUC');axs[1].set_title('Known radial deviations')
axs[1].tick_params(axis='x',labelrotation=25)
fig.tight_layout();fig.savefig(OUT/'toy_geometry.png',dpi=180);plt.close(fig)
print('КОНТРОЛЬНАЯ ГЕОМЕТРИЧЕСКАЯ ДЕМОНСТРАЦИЯ (НЕ AI DETECTION):')
display(toy_metrics.round(3))
export_zip()

Построение local PCA:   0%|          | 0/400 [00:00<?, ?it/s]

КОНТРОЛЬНАЯ ГЕОМЕТРИЧЕСКАЯ ДЕМОНСТРАЦИЯ (НЕ AI DETECTION):


,method,roc_auc
0,Mahalanobis,0.584
1,kNN,1.000
2,Global_PCA,0.518
3,Local_PCA,1.000


Результаты: /kaggle/working/geo_fast_55min_results.zip


PosixPath('/kaggle/working/geo_fast_55min_results.zip')

## 3. Поиск Tiny GenImage, реальный train/cal/test без пересечения

In [47]:
ALIASES={
    'imagenet_ai_0419_biggan':'BigGAN',
    'imagenet_ai_0419_vqdm':'VQDM',
    'imagenet_ai_0424_sdv5':'SD1.5',
    'imagenet_ai_0424_wukong':'Wukong',
    'imagenet_ai_0508_adm':'ADM',
    'imagenet_glide':'GLIDE',
    'imagenet_midjourney':'Midjourney',
}
EXT={'.jpg','.jpeg','.png','.webp','.bmp'}
assert ROOT.exists(),f'Нет {ROOT}: подключите Kaggle датасет Tiny GenImage.'
roots={}
for path in ROOT.rglob('*'):
    if path.is_dir() and path.name in ALIASES and (path/'train'/'nature').is_dir() and (path/'val'/'ai').is_dir():
        roots.setdefault(ALIASES[path.name],path)
assert roots,'Не найдены папки Tiny GenImage /train/nature и /val/ai. Подключите датасет в Add Input.'
GENS=[g for g in GENERATORS_PRIORITY if g in roots]
print('Найдены генераторы:',GENS)

def list_imgs(folder):
    return sorted(x for x in folder.rglob('*') if x.is_file() and x.suffix.lower() in EXT)

# Одно и то же настоящее изображение часто копируется для нескольких генераторов.
# Дедуплицируем по имени, затем по содержимому только выбранных файлов.
by_name_train={}
by_name_val={}
for g,path in roots.items():
    for p in list_imgs(path/'train'/'nature'):
        by_name_train.setdefault(p.name.lower(),p)
    for p in list_imgs(path/'val'/'nature'):
        by_name_val.setdefault(p.name.lower(),p)
train_candidates=list(by_name_train.values())
val_candidates=[p for name,p in by_name_val.items() if name not in by_name_train]
train_candidates=sorted(train_candidates,key=str)
val_candidates=sorted(val_candidates,key=str)
rng=np.random.default_rng(SEED)
rng.shuffle(train_candidates);rng.shuffle(val_candidates)

seen_hash=set()
def unique_take(candidates, count):
    found=[]
    for p in candidates:
        h=hashlib.sha256(p.read_bytes()).hexdigest()
        if h not in seen_hash:
            seen_hash.add(h);found.append(p)
        if len(found)>=count:break
    return found

train_real=unique_take(train_candidates,TRAIN_N)
cal_real=unique_take(val_candidates,CAL_N)
# Именно тот же val-пул, но хеши из train/cal исключаются.
test_real=unique_take(val_candidates,TEST_REAL_N)
fake_by_gen={g:list_imgs(roots[g]/'val'/'ai')[:FAKE_PER_GEN] for g in GENS}
assert len(train_real)>=max(PCA_RANK+1,LOCAL_K+1),f'Не хватает train real: {len(train_real)}'
assert len(cal_real)>=30 and len(test_real)>=30, ('Не хватает val real',len(cal_real),len(test_real))
assert all(len(fake_by_gen[g])>=30 for g in GENS),'Не хватает fake в одном из генераторов'
manifest=[dict(split=s,label=0,generator='real',path=str(p))
          for s,seq in [('train',train_real),('cal',cal_real),('test',test_real)] for p in seq]
manifest += [dict(split='test',label=1,generator=g,path=str(p))
             for g,arr in fake_by_gen.items() for p in arr]
pd.DataFrame(manifest).to_csv(OUT/'manifest.csv',index=False)
print('Real:',{ 'train':len(train_real),'cal':len(cal_real),'test':len(test_real)})
print('Fake test:',{g:len(paths) for g,paths in fake_by_gen.items()})
print('Время от начала:',round(elapsed_min(),1),'мин')
export_zip()

Найдены генераторы: ['BigGAN', 'SD1.5', 'Midjourney', 'VQDM', 'Wukong', 'ADM', 'GLIDE']
Real: {'train': 200, 'cal': 80, 'test': 80}
Fake test: {'BigGAN': 80, 'SD1.5': 80, 'Midjourney': 80, 'VQDM': 80, 'Wukong': 80, 'ADM': 80, 'GLIDE': 80}
Время от начала: 0.2 мин
Результаты: /kaggle/working/geo_fast_55min_results.zip


PosixPath('/kaggle/working/geo_fast_55min_results.zip')

## 4. Один замороженный DINOv2-S и кэш эмбеддингов (только GPU0)

In [48]:
assert torch.cuda.is_available(), ('Для реальных изображений включите Kaggle Accelerator: GPU. '
                                   'Синтетическая демонстрация выше уже сохранена.')
DEVICE='cuda:0'
from transformers import AutoModel
print('Загружаем замороженный',MODEL_ID, 'на',DEVICE)
model=AutoModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
model.requires_grad_(False)
print('Число параметров:',sum(p.numel() for p in model.parameters()))
MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32).view(3,1,1)
STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32).view(3,1,1)

class ImageSet(Dataset):
    def __init__(self,paths,cond):self.paths=list(paths);self.cond=cond
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        p=self.paths[index]
        with Image.open(p) as im:
            x=ImageOps.fit(ImageOps.exif_transpose(im).convert('RGB'),(224,224),
                           method=Image.Resampling.BICUBIC)
        if self.cond=='jpeg60':
            buf=io.BytesIO();x.save(buf,format='JPEG',quality=60);buf.seek(0)
            with Image.open(buf) as im2: x=im2.convert('RGB').copy()
        elif self.cond!='clean':raise ValueError(self.cond)
        arr=np.array(x,dtype=np.uint8,copy=True)
        x=torch.from_numpy(arr).permute(2,0,1).to(torch.float32).div(255.0)
        return (x-MEAN)/STD

@torch.inference_mode()
def embed(paths,cond='clean',label=''):
    if not paths: return np.empty((0,384),dtype=np.float32)
    digest=hashlib.sha256(('\n'.join(map(str,paths))+'|'+cond+'|'+MODEL_ID+'|224-v1').encode()).hexdigest()[:20]
    fn=CACHE/f'features_{digest}.npy'
    if fn.exists():
        arr=np.load(fn)
        if arr.shape==(len(paths),384):
            print('Кэш признаков:',label,len(paths))
            return arr
    dl=DataLoader(ImageSet(paths,cond),batch_size=BATCH,shuffle=False,
                  num_workers=0,pin_memory=False,drop_last=False)
    parts=[]
    for batch in tqdm(dl,desc=f'DINO {label} {cond}',leave=False):
        batch=batch.to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16):
            z=model(pixel_values=batch).last_hidden_state[:,0,:]
            z=F.normalize(z.float(),dim=1)
        parts.append(z.cpu().numpy().astype(np.float32))
    arr=np.concatenate(parts,axis=0)
    np.save(fn,arr)
    return arr

features_train=embed(train_real,label='train real')
features_cal=embed(cal_real,label='cal real')
features_test=embed(test_real,label='test real')
print('Эмбеддинги готовы:',features_train.shape,features_cal.shape,features_test.shape,
      f'{elapsed_min():.1f} мин с начала')

Загружаем замороженный facebook/dinov2-small на cuda:0
Число параметров: 22056576


DINO train real clean:   0%|          | 0/7 [00:00<?, ?it/s]

DINO cal real clean:   0%|          | 0/3 [00:00<?, ?it/s]

DINO test real clean:   0%|          | 0/3 [00:00<?, ?it/s]

Эмбеддинги готовы: (200, 384) (80, 384) (80, 384) 0.3 мин с начала


## 5. Подгонка статистики только по train real и калибровка на cal real

In [49]:
print('Строим статистические модели (никаких fake при обучении)')
scorer=GeometryScorers(features_train,p_global=PCA_RANK,
                       local_k=LOCAL_K,p_local=LOCAL_RANK,knn_k=KNN_K)
cal_scores=scorer.score(features_cal)
THRESHOLDS={name:float(np.quantile(s,.95,method='higher')) for name,s in cal_scores.items()}
write_json('thresholds_real_only.json',THRESHOLDS)
write_json('pca_diagnostics.json',dict(global_explained_variance_ratio=scorer.ratio,
                                      global_rank=PCA_RANK,local_rank=LOCAL_RANK,local_k=LOCAL_K))
print('Объяснённая дисперсия глобального PCA:',f'{scorer.ratio:.1%}')
print('Пороги по реальным (ожидаемый калибровочный FPR ≈5%):',THRESHOLDS)
export_zip()

Строим статистические модели (никаких fake при обучении)


Построение local PCA:   0%|          | 0/200 [00:00<?, ?it/s]

Объяснённая дисперсия глобального PCA: 40.9%
Пороги по реальным (ожидаемый калибровочный FPR ≈5%): {'Mahalanobis': 414.1042693225949, 'kNN': 1.3084373950958252, 'Global_PCA': 0.9635332226753235, 'Local_PCA': 0.9985222816467285}
Результаты: /kaggle/working/geo_fast_55min_results.zip


PosixPath('/kaggle/working/geo_fast_55min_results.zip')

## 6. Настоящий тест: ROC-AUC по генераторам (сохранение **после каждого генератора**)

Сначала обрабатывается **clean**, и только потом JPEG60. Поэтому даже если до часа останется мало времени, в `metrics.csv` будут результаты clean. Ранние метрики не корректируются по информации о fake, а направленность всех scores задана заранее (выше score → более подозрительный объект). Нулевая гипотеза — ROC-AUC=0.5.

In [50]:
rows=[]
score_rows=[]
completed=[]

for condition in CONDITIONS:
    if budget_left()<3:
        print('Лимит времени близок; пропускаем',condition)
        break
    if condition=='clean':
        feat_real=features_test
    else:
        feat_real=embed(test_real,cond=condition,label='test real')
    real_sc=scorer.score(feat_real)
    for gen in GENS:
        if budget_left()<2.0:
            print('Достигнут лимит времени — останавливаемся после уже сохранённых результатов')
            break
        paths=fake_by_gen[gen]
        feature_fake=embed(paths,cond=condition,label=gen)
        fake_sc=scorer.score(feature_fake)
        for method in METHODS:
            r,f=real_sc[method],fake_sc[method]
            y=np.r_[np.zeros(len(r)),np.ones(len(f))]
            auc=float(roc_auc_score(y,np.r_[r,f]))
            low,high=auc_ci(r,f,seed=SEED+sum(map(ord,gen))+sum(map(ord,method)),B=BOOTSTRAP) if condition=='clean' else (float('nan'),float('nan'))
            thr=THRESHOLDS[method]
            rows.append(dict(condition=condition,generator=gen,method=method,
                             roc_auc=auc,ci_lower=low,ci_upper=high,
                             tpr_at_cal_fpr5=float(np.mean(f>thr)),
                             observed_test_fpr=float(np.mean(r>thr)),
                             n_real=len(r),n_fake=len(f),threshold=thr))
            score_rows += [dict(condition=condition,generator=gen,method=method,label=1,
                                path=str(p),score=float(s)) for p,s in zip(paths,f)]
            if gen==GENS[0]:
                score_rows += [dict(condition=condition,generator='real',method=method,label=0,
                                    path=str(p),score=float(s)) for p,s in zip(test_real,r)]
        completed.append((condition,gen))
        metrics=pd.DataFrame(rows)
        metrics.to_csv(OUT/'metrics.csv',index=False)
        pd.DataFrame(score_rows).to_csv(OUT/'scores_per_image.csv',index=False)
        save_metric_plots(metrics)
        print(f'ГОТОВО {condition} | {gen} | ',
              ', '.join(f'{m}: {metrics[(metrics.generator==gen)&(metrics.condition==condition)&(metrics.method==m)].roc_auc.iloc[0]:.3f}' for m in METHODS))
        export_zip()
    if budget_left()<2.0:break

metrics=pd.DataFrame(rows)
if not metrics.empty:
    print('\nИТОГ clean (ROC-AUC):')
    display(save_metric_plots(metrics).round(3))
    print('\nРеализованная FPR на held-out real отличается от целевых 5% из-за конечной выборки.')
print('Групп обработано:',completed,'; elapsed:',round(elapsed_min(),1),'мин')

DINO BigGAN clean:   0%|          | 0/3 [00:00<?, ?it/s]

ГОТОВО clean | BigGAN |  Mahalanobis: 0.492, kNN: 0.528, Global_PCA: 0.525, Local_PCA: 0.548
Результаты: /kaggle/working/geo_fast_55min_results.zip


DINO SD1.5 clean:   0%|          | 0/3 [00:00<?, ?it/s]

ГОТОВО clean | SD1.5 |  Mahalanobis: 0.570, kNN: 0.567, Global_PCA: 0.558, Local_PCA: 0.578
Результаты: /kaggle/working/geo_fast_55min_results.zip


DINO Midjourney clean:   0%|          | 0/3 [00:00<?, ?it/s]

ГОТОВО clean | Midjourney |  Mahalanobis: 0.363, kNN: 0.353, Global_PCA: 0.376, Local_PCA: 0.397
Результаты: /kaggle/working/geo_fast_55min_results.zip


... (21 outputs left out)

DINO GLIDE jpeg60:   0%|          | 0/3 [00:00<?, ?it/s]

ГОТОВО jpeg60 | GLIDE |  Mahalanobis: 0.402, kNN: 0.435, Global_PCA: 0.426, Local_PCA: 0.473
Результаты: /kaggle/working/geo_fast_55min_results.zip

ИТОГ clean (ROC-AUC):


method,Mahalanobis,kNN,Global_PCA,Local_PCA
generator,,,,
BigGAN,0.492,0.528,0.525,0.548
SD1.5,0.570,0.567,0.558,0.578
Midjourney,0.363,0.353,0.376,0.397
VQDM,0.452,0.509,0.498,0.523
Wukong,0.457,0.484,0.467,0.507
ADM,0.393,0.376,0.401,0.419
GLIDE,0.423,0.442,0.449,0.473



Реализованная FPR на held-out real отличается от целевых 5% из-за конечной выборки.
Групп обработано: [('clean', 'BigGAN'), ('clean', 'SD1.5'), ('clean', 'Midjourney'), ('clean', 'VQDM'), ('clean', 'Wukong'), ('clean', 'ADM'), ('clean', 'GLIDE'), ('jpeg60', 'BigGAN'), ('jpeg60', 'SD1.5'), ('jpeg60', 'Midjourney'), ('jpeg60', 'VQDM'), ('jpeg60', 'Wukong'), ('jpeg60', 'ADM'), ('jpeg60', 'GLIDE')] ; elapsed: 1.5 мин


## 7. Дополнительный результат: сравнение Local PCA vs Mahalanobis с парным CI (без дополнительного DINO)

In [51]:
if 'metrics' in globals() and not metrics.empty:
    # Пары выровнены по тем же изображениям; bootstrap одинаковых индексов для двух методов.
    sf=pd.DataFrame(score_rows)
    comparison=[]
    for gen in [g for g in GENS if (('clean',g) in completed)]:
        dd={}
        for m in ('Local_PCA','Mahalanobis'):
            a=sf[(sf.condition=='clean')&(sf.generator=='real')&(sf.method==m)].sort_values('path').score.to_numpy()
            b=sf[(sf.condition=='clean')&(sf.generator==gen)&(sf.method==m)].sort_values('path').score.to_numpy()
            dd[m]=(a,b)
        if not len(dd['Local_PCA'][0]) or not len(dd['Local_PCA'][1]): continue
        rng=np.random.default_rng(SEED+sum(map(ord,gen)))
        ar,af=dd['Local_PCA'];br,bf=dd['Mahalanobis']
        deltas=[]
        for _ in range(BOOTSTRAP):
            ir=rng.integers(0,len(ar),len(ar));jf=rng.integers(0,len(af),len(af))
            y=np.r_[np.zeros(len(ir)),np.ones(len(jf))]
            dl=roc_auc_score(y,np.r_[ar[ir],af[jf]])
            dm=roc_auc_score(y,np.r_[br[ir],bf[jf]])
            deltas.append(dl-dm)
        comparison.append(dict(generator=gen,
                 delta_auc_local_minus_maha=float(roc_auc_score(
                     np.r_[np.zeros(len(ar)),np.ones(len(af))],np.r_[ar,af])-roc_auc_score(
                     np.r_[np.zeros(len(br)),np.ones(len(bf))],np.r_[br,bf])),
                 ci_lower=float(np.quantile(deltas,.025)),ci_upper=float(np.quantile(deltas,.975))))
    comparison=pd.DataFrame(comparison)
    comparison.to_csv(OUT/'local_vs_maha_bootstrap.csv',index=False)
    print('Парная разность ROC-AUC Local PCA минус Mahalanobis:')
    display(comparison.round(3))
    export_zip()

Парная разность ROC-AUC Local PCA минус Mahalanobis:


,generator,delta_auc_local_minus_maha,ci_lower,ci_upper
0,BigGAN,0.055,0.008,0.109
1,SD1.5,0.008,-0.050,0.063
2,Midjourney,0.034,-0.007,0.078
3,VQDM,0.072,0.021,0.121
4,Wukong,0.050,0.003,0.108
5,ADM,0.026,-0.020,0.078
6,GLIDE,0.050,-0.003,0.109


Результаты: /kaggle/working/geo_fast_55min_results.zip


## 8. Готовый вывод для слайда и скачивание

In [52]:
text_lines=[
    '# Мини-эксперимент: геометрическое описание DINOv2-признаков реальных изображений',
    f'Бюджет: {TIME_LIMIT_MIN} минут; фактическое время на момент отчёта: {elapsed_min():.1f} минут.',
    f'Real train: {len(train_real)}, cal: {len(cal_real)}, test: {len(test_real)}.',
    f'Замороженный {MODEL_ID}; PCA global rank={PCA_RANK}, local rank={LOCAL_RANK}, k={LOCAL_K}.',
    f'Глобальный PCA объясняет {scorer.ratio:.1%} дисперсии train-real.',
    'Модель не обучается на изображениях AI-генераторов (real-only).',
    'Проверенные методы: '+', '.join(METHODS),
    'Важно: Synthetic toy-окружность демонстрирует геометрию, но не качество детекции AI.',
    'Важно: нет проверенного semantic matching между real и fake, результаты пилотные.',
    'Важно: это НЕ REM, и никакая граница многообразия в строгом смысле не восстановлена.',
    '', '## ROC-AUC clean по генераторам',
]
if 'metrics' in globals() and not metrics.empty:
    txt=metrics[metrics.condition=='clean'].pivot(index='generator',columns='method',values='roc_auc')
    text_lines.append(txt.to_string(float_format=lambda x:f'{x:.3f}'))
else:text_lines.append('Выполнен только toy-эксперимент.')
(OUT/'report.md').write_text('\n'.join(text_lines)+'\n',encoding='utf-8')
print('\n'.join(text_lines))
zip_path=export_zip()
display(FileLink(str(zip_path)))
print('Для доклада лучше использовать auc_clean_heatmap.png и конкретную строку таблицы metrics.csv.')

# Мини-эксперимент: геометрическое описание DINOv2-признаков реальных изображений
Бюджет: 55 минут; фактическое время на момент отчёта: 1.6 минут.
Real train: 200, cal: 80, test: 80.
Замороженный facebook/dinov2-small; PCA global rank=32, local rank=8, k=32.
Глобальный PCA объясняет 40.9% дисперсии train-real.
Модель не обучается на изображениях AI-генераторов (real-only).
Проверенные методы: Mahalanobis, kNN, Global_PCA, Local_PCA
Важно: Synthetic toy-окружность демонстрирует геометрию, но не качество детекции AI.
Важно: нет проверенного semantic matching между real и fake, результаты пилотные.
Важно: это НЕ REM, и никакая граница многообразия в строгом смысле не восстановлена.

## ROC-AUC clean по генераторам
method      Global_PCA  Local_PCA  Mahalanobis   kNN
generator                                           
ADM              0.401      0.419        0.393 0.376
BigGAN           0.525      0.548        0.492 0.528
GLIDE            0.449      0.473        0.423 0.442
Midjourney    

/kaggle/working/geo_fast_55min_results.zip

Для доклада лучше использовать auc_clean_heatmap.png и конкретную строку таблицы metrics.csv.
